## Vector Store Retriever
A vector store retriever in LanhChain is the most common type of retriever that lets you search and fetch documents from a vector store based on semantic similarity using vector embedding

In [14]:
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document

In [15]:
# Step 1: Your source documents
documents = [
    Document(page_content="LangChain is a framework for building applications with large language models (LLMs). It provides tools and abstractions to simplify the process of integrating LLMs into your applications.", metadata={"source": "doc1"}),
    Document(page_content="Chroma is an open-source vector database that allows you to store and query embeddings efficiently. It is designed to work seamlessly with machine learning models and provides fast similarity search capabilities.", metadata={"source": "doc2"}),
    Document(page_content="Hugging Face is a company that provides a platform for sharing and deploying machine learning models, particularly in the field of natural language processing (NLP). They offer a wide range of pre-trained models and tools for building NLP applications.", metadata={"source": "doc3"}),     
    Document(page_content="FAISS (Facebook AI Similarity Search) is a library developed by Facebook AI Research for efficient similarity search and clustering of dense vectors. It is widely used in machine learning and information retrieval applications.", metadata={"source": "doc4"}),
    Document(page_content="Wikipedia is a free online encyclopedia that contains a vast amount of information on a wide range of topics. It is collaboratively edited by volunteers from around the world and is one of the most popular reference websites on the internet.", metadata={"source": "doc5"}),
]

In [16]:
# Step 2: Initialize the embedding model
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

In [17]:
# Step 3: Create Chroma vector store in memory
vectorstore = Chroma.from_documents(
    documents=documents,
    embedding=embeddings,
    collection_name="02_vector_store_retriever_collection"
)

In [18]:
# Step 4: Convert vectorstore into a retriever
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

In [22]:
# query

query = "What is Chroma?"
results = retriever.invoke(query)

/mnt/d/Academics/Generative AI by CampusX/.venv/lib/python3.12/site-packages/torch/nn/modules/module.py:1784: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


In [23]:
for i, doc in enumerate(results):
    print(f"Document {i+1}:")
    print(f"Title: {doc.metadata.get('title', 'N/A')}")
    print(f"Content: {doc.page_content[:500]}...")  # Print first 500 characters
    print(f"Content: {doc.page_content}") # Print full article
    print("-" * 80)

Document 1:
Title: N/A
Content: Chroma is an open-source vector database that allows you to store and query embeddings efficiently. It is designed to work seamlessly with machine learning models and provides fast similarity search capabilities....
Content: Chroma is an open-source vector database that allows you to store and query embeddings efficiently. It is designed to work seamlessly with machine learning models and provides fast similarity search capabilities.
--------------------------------------------------------------------------------
Document 2:
Title: N/A
Content: Chroma is an open-source vector database that allows you to store and query embeddings efficiently. It is designed to work seamlessly with machine learning models and provides fast similarity search capabilities....
Content: Chroma is an open-source vector database that allows you to store and query embeddings efficiently. It is designed to work seamlessly with machine learning models and provides fast similarity 

In [24]:
vector_store_result = vectorstore.similarity_search(query, k=2)

/mnt/d/Academics/Generative AI by CampusX/.venv/lib/python3.12/site-packages/torch/nn/modules/module.py:1784: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


In [26]:
# print(vector_store_result)
for i, doc in enumerate(vector_store_result):
    print(f"Document {i+1}:")
    print(f"Title: {doc.metadata.get('title', 'N/A')}")
    print(f"Content: {doc.page_content[:500]}...")  # Print first 500 characters
    print(f"Content: {doc.page_content}") # Print full article
    print("-" * 80)

Document 1:
Title: N/A
Content: Chroma is an open-source vector database that allows you to store and query embeddings efficiently. It is designed to work seamlessly with machine learning models and provides fast similarity search capabilities....
Content: Chroma is an open-source vector database that allows you to store and query embeddings efficiently. It is designed to work seamlessly with machine learning models and provides fast similarity search capabilities.
--------------------------------------------------------------------------------
Document 2:
Title: N/A
Content: Chroma is an open-source vector database that allows you to store and query embeddings efficiently. It is designed to work seamlessly with machine learning models and provides fast similarity search capabilities....
Content: Chroma is an open-source vector database that allows you to store and query embeddings efficiently. It is designed to work seamlessly with machine learning models and provides fast similarity 

## Differnece between vectorstore_Retriever and vectorstore.simliarity_search

---

### 1️⃣ `vectorstore.similarity_search(query, k=4)`

* **Direct method on the vector store**.
* You pass a query (string or embedding), it searches the vector index, and **returns the top-k most similar documents**.
* Returns a **list of `Document` objects**.

✅ Example:

```python
results = vectorstore.similarity_search("Who is a wicket-keeper?", k=2)
for doc in results:
    print(doc.page_content, doc.metadata)
```

---

### 2️⃣ `vectorstore.as_retriever()` (or `vectorstore_Retriever`)

* **Abstraction layer** on top of the vector store.
* Returns a **Retriever object** instead of directly querying the store.
* Retrievers define a standard interface (`get_relevant_documents(query)`) used in **chains, agents, and RAG pipelines**.
* You can configure search type: `"similarity"`, `"mmr"`, `"similarity_score_threshold"`, etc.

✅ Example:

```python
retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 2})
results = retriever.get_relevant_documents("Who is a wicket-keeper?")
for doc in results:
    print(doc.page_content, doc.metadata)
```

---

### 🔑 **Key Differences**

| Feature | `similarity_search()`           | `as_retriever()`                            |
| ------- | ------------------------------- | ------------------------------------------- |
| Returns | List of `Document`s             | `Retriever` object                          |
| Usage   | Direct queries to the vector DB | Plug into **LangChain Chains/Agents**       |
| Config  | Basic `k`                       | Advanced (`k`, score thresholds, MMR, etc.) |
| Purpose | Standalone searches             | Integrations in RAG pipelines               |

---

👉 **Rule of thumb:**

* Use `similarity_search()` if you just want to quickly search your vector DB.
* Use `as_retriever()` if you’re building **chains, QA systems, or RAG apps** where retrieval is a modular step.

---